# File and E-mail System Automation using Python

This notebook accompanies Topic 9 and demonstrates **working, practical examples** of file system and e-mail automation in Python.


## Why File System and E-mail Automation Matters

Automation reduces repetitive manual work and makes workflows reproducible and less error-prone. Often we have some file or e-mail related tasks that could benefit from programmed approach.


## Part I – File System Automation

### Searching and Filtering Files

In [1]:

from pathlib import Path

root = Path(".")
files = list(root.rglob("*.txt"))

print(f"Found {len(files)} text files")
for f in files[:5]:
    print(f)


Found 0 text files


### Bulk Copying Files

In [ ]:

import shutil
from pathlib import Path

src = Path(".")
dst = Path("collected_txt")
dst.mkdir(exist_ok=True)

for f in src.rglob("*.txt"):
    shutil.copy2(f, dst / f.name)

print("Files copied.")


### Bulk Renaming Files

In [ ]:

from pathlib import Path

folder = Path("collected_txt")

for i, f in enumerate(folder.glob("*.txt"), start=1):
    new_name = folder / f"document_{i:03d}.txt"
    f.rename(new_name)

print("Files renamed.")


## Part II – E-mail Automation

### Sending E-mail via SMTP

Sending e-mails programmatically is a common automation task. Below is a practical guide to sending e-mails via Gmail using SMTP in Python.

## Sending E-mail via Gmail using SMTP (Practical Guide)

This example demonstrates **the most reliable and realistic way to automate e-mail sending in 2025** using Python and Gmail: **SMTP with an App Password**.  
It avoids OAuth, browser pop-ups, and token refresh logic, making it suitable for scripts, notebooks, and scheduled jobs.

---

### What This Example Does

- Connects securely to Gmail’s SMTP server
- Authenticates using an **App Password**
- Sends a plain-text e-mail
- Uses **environment variables** so credentials are not hard-coded

This is ideal for:
- Notifications after scripts finish
- Sending reports or logs
- Automated alerts from data processing pipelines

---

### Prerequisites

#### 1. A Gmail Account
Any regular `@gmail.com` account works.

#### 2. Two-Step Verification Enabled
App passwords are **only available if 2-Step Verification is enabled**.

- Google Account → Security → Signing in to Google
- Enable **2-Step Verification**

#### 3. Create an App Password
1. Go to: Google Account → Security → App passwords
1b. Alternative go directly to: https://myaccount.google.com/apppasswords WHILE LOGGED IN ! to your Google account.
2. App: **Mail**
3. Device: **Other (Custom)** → name it e.g. `Python SMTP`
4. Copy the **16-character password**

⚠️ This password is **not** your Gmail password.

---

### Setting Credentials Securely (Required)

Credentials must **not** be stored directly in code.

#### On Windows (PowerShell)

```powershell
setx SMTP_EMAIL "your_email@gmail.com"
setx SMTP_PASSWORD "your_16_char_app_password"
```

Restart VS Code / Jupyter after setting variables.

#### On macOS / Linux (bash / zsh)

```bash
export SMTP_EMAIL="your_email@gmail.com"
export SMTP_PASSWORD="your_16_char_app_password"
```

---

### How the Code Works (Step by Step)

1. **Load credentials from environment variables**

```python
EMAIL_ADDRESS = os.getenv("SMTP_EMAIL")
EMAIL_PASSWORD = os.getenv("SMTP_PASSWORD")
```

This keeps secrets out of:
- source code
- Git repositories
- shared notebooks

---

2. **Create an e-mail message**

```python
msg = EmailMessage()
msg["Subject"] = "Automated Test Email"
msg["From"] = EMAIL_ADDRESS
msg["To"] = EMAIL_ADDRESS
msg.set_content("This is a test email sent from Python automation.")
```

---

3. **Connect to Gmail securely**

```python
with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
```

- `SMTP_SSL` creates an encrypted connection
- Port **465** is Gmail’s secure SMTP port

---

4. **Authenticate and send**

```python
server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
server.send_message(msg)
```

---

### Common Errors and Fixes

| Error | Likely Cause |
|------|-------------|
| `Authentication failed` | App password incorrect or missing |
| `SMTPAuthenticationError` | 2-Step Verification not enabled |
| `NoneType` email address | Environment variables not set |
| Works in terminal, not in Jupyter | Kernel not restarted |

---

### Security Notes (Important)

- App passwords grant **limited access** and can be revoked anytime
- Never commit passwords to GitHub
- Use SMTP only for **sending**, not inbox management
- For reading e-mail, Gmail API with OAuth is required

---

### Why SMTP Is Still Used in 2025

- Universally supported
- Works with Gmail, Outlook, and university mail
- Stable for automation and cron jobs
- Minimal setup compared to full e-mail APIs

This is why SMTP is the **recommended approach** for this course.


In [1]:

import smtplib
from email.message import EmailMessage
import os

EMAIL_ADDRESS = os.getenv("SMTP_EMAIL")
EMAIL_PASSWORD = os.getenv("SMTP_PASSWORD")

msg = EmailMessage()
msg["Subject"] = "Automated Test Email"
msg["From"] = EMAIL_ADDRESS
msg["To"] = EMAIL_ADDRESS
msg.set_content("This is a test email sent from Python automation.")

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
    server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
    server.send_message(msg)

print("Email sent successfully.")


Email sent successfully.


In [ ]:
# let's simply save SENT_TO_EMAIL to a variable for reuse
# let's prompt the user for the recipient email address
SENT_TO_EMAIL = input("Enter the recipient email address: ")
# print(f"Set SENT_TO_EMAIL to: {SENT_TO_EMAIL}")

### Sending E-mail with Attachment

In [3]:

from email.message import EmailMessage
import smtplib
from pathlib import Path
import os

EMAIL_ADDRESS = os.getenv("SMTP_EMAIL")
EMAIL_PASSWORD = os.getenv("SMTP_PASSWORD")

attachment = Path("example.txt")
attachment.write_text("Example attachment file")

msg = EmailMessage()
msg["Subject"] = "Email with Attachment"
msg["From"] = EMAIL_ADDRESS
msg["To"] = SENT_TO_EMAIL
msg.set_content("See attached file.")

msg.add_attachment(
    attachment.read_bytes(),
    maintype="text",
    subtype="plain",
    filename=attachment.name
)

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
    server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
    server.send_message(msg)

print("Email with attachment sent.")


Email with attachment sent.


## Gmail API (Conceptual Example)



This section explains **what “full Gmail API access” actually means**, why it requires **OAuth**, and what the **main setup steps** are.  
No code is shown here on purpose — the goal is conceptual clarity, not implementation detail.

---

## Why Gmail API Requires OAuth (and SMTP Does Not)

### SMTP (what we already did)
- Old, protocol-based authentication
- Username + password (or app password)
- Designed for **sending mail only**
- Limited scope and limited control

### Gmail API (modern approach)
- Direct programmatic access to Gmail as a service
- Can **read**, **search**, **modify**, and **send** mail
- Must respect modern security and privacy rules
- Requires **explicit, granular user consent**

Because Gmail API can expose **the entire mailbox**, Google does **not** allow password-based access.  
Instead, it uses **OAuth 2.0**, an industry-standard authorization framework.

---

## What OAuth Is (Plain Explanation)

**OAuth is not authentication — it is authorization.**

In simple terms:

> “I allow this application to do *these specific things* with my account,  
> without giving it my password.”

OAuth achieves this by:
- Separating **identity** (Google login) from **permissions**
- Issuing **tokens** instead of passwords
- Allowing permissions to be **revoked at any time**

---

## OAuth Roles (Who Does What)

There are three actors:

1. **User**
   - Owns the Gmail account
   - Grants or denies access

2. **Application (your script / notebook)**
   - Requests permission
   - Never sees the Gmail password

3. **Google Authorization Server**
   - Verifies identity
   - Issues and validates tokens

---

## What OAuth Tokens Are

OAuth uses **tokens**, not passwords:

- **Access token**
  - Short-lived (minutes)
  - Used for actual API calls

- **Refresh token**
  - Long-lived
  - Used to obtain new access tokens automatically

Your script:
- Stores tokens locally
- Refreshes access silently
- Does **not** re-prompt the user every run

---

## What “Full Gmail API Access” Enables

Depending on granted scopes, an application can:

- Read inbox and sent mail
- Search messages using Gmail query syntax
- Download attachments
- Send new messages
- Reply to existing threads
- Apply or remove labels
- Mark messages read/unread

This power is exactly why OAuth is mandatory.

---

## High-Level OAuth Setup Flow (Google Cloud Console)

### Step 1: Create a Google Cloud Project
- A project is a **security and billing boundary**
- Recommended to create a dedicated project for demos or courses

Purpose:
- Isolate credentials
- Simplify cleanup
- Avoid mixing unrelated APIs

---

### Step 2: Enable Gmail API
- APIs are disabled by default
- Explicitly enabling Gmail API:
  - Signals intent
  - Activates quota tracking
  - Makes credentials usable

---

### Step 3: Configure OAuth Consent Screen

This is the **human-facing** part of OAuth.

You must define:
- Application name
- Support contact email
- Developer contact email
- Requested scopes (permissions)

Key choice:
- **External** → personal Gmail accounts (most teaching use cases)
- **Internal** → Google Workspace domains only

For teaching and demos:
- App usually stays in **Testing mode**
- Only specified test users can authorize it
- No public verification required

---

### Step 4: Choose Scopes (Permissions)

Scopes define **what the app can do**.

Examples:
- Read-only access
- Send-only access
- Modify mailbox state

Principle:
> Always request the **smallest set of scopes** that solves the problem.

Over-scoping:
- Triggers stronger warnings
- Raises user distrust
- May be blocked by institutional policy

---

### Step 5: Create OAuth Client Credentials

This step produces:
- A **Client ID**
- A **Client Secret**
- Stored in a credentials JSON file

Important:
- This file identifies *your app*, not the user
- It must be kept private
- It is not a password, but still sensitive

For local automation:
- **Desktop Application** client type is typically used

---

## What Happens When the Script Runs (Conceptually)

1. Script detects no valid token
2. Browser opens Google login page
3. User logs in and sees consent screen
4. User approves requested scopes
5. Google redirects back with authorization result
6. Script receives tokens
7. Tokens are stored locally
8. Future runs reuse tokens silently

This is usually a **one-time interaction per machine**.

---

## Token Storage and Security Considerations

Token files:
- Act like **keys**
- Grant access without re-login

Best practices:
- Do not commit token files to Git
- Add them to `.gitignore`
- Revoke tokens if a machine is lost
- Use separate projects for teaching vs production

Revocation is always possible via Google Account settings.

---

## Institutional and University Account Caveats

For Google Workspace (university) accounts:
- OAuth scopes may be restricted
- External apps may be blocked
- Admin approval may be required
- Some APIs may be disabled entirely

This is why:
- **Personal Gmail accounts** are strongly recommended for teaching
- SMTP is preferred for simple automation examples

---

## Why Gmail API Is Usually “Advanced” Material

Compared to SMTP:
- More setup steps
- More security concepts
- More ways to fail due to policy
- More responsibility for data handling

However:
- It is the **correct professional solution**
- It scales
- It respects modern security models

---

## Conceptual Takeaway

- SMTP = simple, pragmatic, limited
- Gmail API + OAuth = powerful, secure, complex
- OAuth exists to protect users, not to annoy developers
- Understanding OAuth is a **transferable skill** beyond Gmail

In practice OAUTH is widely used across many APIs and services today, BUT many pitfalls and complexities exist.ches remain, so SMTP is often the preferred choice for simple automation tasks.


## Outlook / Exchange API (Conceptual Example)

This section explains **how full Outlook / Exchange mailbox automation works conceptually**, why it relies on **OAuth and Microsoft Graph**, and what the **main setup steps** are.  
As with Gmail API, no code is shown here on purpose — the goal is architectural understanding rather than implementation.

---

## Why Outlook / Exchange Automation Is Different

Modern Outlook accounts (including most university and enterprise mailboxes) are part of **Microsoft 365** and backed by **Exchange Online**.

Key consequences:
- There is **no password-based API access**
- SMTP is limited to sending mail only
- Full mailbox access requires **Microsoft Graph API**
- Authentication is **OAuth 2.0 via Azure Active Directory (Microsoft Entra ID)**

In short:  
> If you want to *read, search, or manage* Outlook mail programmatically, you must use Microsoft Graph and OAuth.

---

## What Microsoft Graph Is

**Microsoft Graph** is Microsoft’s unified API layer for Microsoft 365 services, including:
- Outlook / Exchange mail
- Calendar
- Contacts
- OneDrive
- Teams
- Users and groups

For e-mail automation, Graph replaces older Exchange-specific APIs.

---

## What OAuth Means in the Microsoft Ecosystem

Conceptually identical to Google OAuth, but implemented through **Microsoft Entra ID** (formerly Azure Active Directory).

OAuth in this context:
- Separates user credentials from application access
- Uses **tokens** instead of passwords
- Enforces permissions via **scopes**
- Allows administrators to control and audit access

---

## Actors in Outlook / Exchange OAuth

1. **User**
   - Owns the mailbox
   - Grants consent (if allowed)

2. **Application**
   - Registered in Microsoft Entra ID
   - Requests permissions

3. **Microsoft Identity Platform**
   - Authenticates users
   - Issues and validates tokens

---

## What Full Outlook / Exchange API Access Enables

Depending on granted permissions, an application can:

- Read inbox and sent mail
- Search messages
- Download attachments
- Send mail on behalf of a user
- Reply and manage threads
- Move, delete, and categorize messages
- Work with folders and rules

This level of access is tightly controlled in enterprise environments.

---

## High-Level Setup Flow (Microsoft 365 / Azure Portal)

### Step 1: Register an Application in Entra ID

- Create an **App Registration**
- This defines:
  - App identity
  - Supported account types
  - Redirect URIs (if applicable)

Purpose:
- Establish a trust boundary
- Enable token issuance

---

### Step 2: Choose Account Types

Options include:
- Single-tenant (one organization)
- Multi-tenant (multiple organizations)
- Personal Microsoft accounts (Outlook.com)

For universities:
- Typically **single-tenant**
- Often restricted to organization-managed apps

---

### Step 3: Configure API Permissions

Permissions are selected from **Microsoft Graph**.

Two major models:

#### Delegated permissions
- App acts **on behalf of a signed-in user**
- User consent is required
- Most common for interactive scripts and tools

#### Application permissions
- App acts **without a user**
- Requires **administrator consent**
- Very powerful
- Rarely allowed in teaching environments

---

### Step 4: Grant Consent

Depending on tenant policy:
- User may grant consent themselves
- Or **administrator approval is required**
- Many universities block user consent entirely

This is the most common point of failure in academic settings.

---

### Step 5: Create Client Credentials

Depending on authentication flow:
- Client secret
- Or certificate-based authentication

These credentials identify the application, not the user.

---

## What Happens When the App Runs (Conceptually)

1. App requests authorization
2. User signs in via Microsoft login
3. Consent screen lists requested permissions
4. Tokens are issued by Microsoft Identity Platform
5. App calls Microsoft Graph using tokens
6. Tokens are refreshed automatically as needed

As with Gmail, this is usually a **one-time consent per user**.

---

## Token and Security Considerations

- Tokens grant access equivalent to granted permissions
- Leaked tokens are a security incident
- Best practices include:
  - Secure storage
  - Short-lived secrets
  - Certificate-based auth for production
  - Explicit permission scoping

Revocation and auditing are handled centrally by administrators.

---

## Institutional Reality (Important for Teaching)

In university / enterprise environments:
- App registration may be locked down
- External apps may be disallowed
- Admin consent may be mandatory
- Personal experimentation may be impossible

This is why:
- SMTP is often the only practical option
- Outlook API examples are usually **conceptual only** in courses

This is not a technical limitation — it is a **policy decision**.

---

## Why Outlook / Exchange API Is Advanced Material

Compared to SMTP:
- Requires identity platform knowledge
- Requires admin cooperation
- Strong security model
- Highly auditable and controlled

However:
- It is the **correct enterprise solution**
- Scales well
- Required for production systems

---

## Conceptual Takeaway

- Outlook / Exchange automation is governed by enterprise security
- Microsoft Graph is the single entry point
- OAuth protects users and organizations
- Institutional policy, not Python, is usually the blocker

Understanding this prepares students to:
- Reason about real-world enterprise APIs
- Communicate effectively with IT departments
- Design systems that respect security boundaries

---

## Authoritative References

### Microsoft Graph & Outlook
- Microsoft Graph Overview  
  https://learn.microsoft.com/graph/overview

- Outlook mail API overview  
  https://learn.microsoft.com/graph/outlook-concept-overview

- Working with Outlook mail via Microsoft Graph  
  https://learn.microsoft.com/graph/api/resources/mail-api-overview

### Authentication & OAuth (Microsoft)
- Microsoft Identity Platform overview  
  https://learn.microsoft.com/entra/identity-platform/v2-overview

- OAuth 2.0 authorization code flow  
  https://learn.microsoft.com/entra/identity-platform/v2-oauth2-auth-code-flow

- App registration in Microsoft Entra ID  
  https://learn.microsoft.com/entra/identity-platform/quickstart-register-app

### Permissions & Consent
- Microsoft Graph permissions reference  
  https://learn.microsoft.com/graph/permissions-reference

- Admin consent workflow  
  https://learn.microsoft.com/entra/identity-platform/admin-consent-workflow


In [ ]:
# how one could read their own e-mails from Outlook using Graph API - example only, not functional without proper setup
import requests
access_token = sys.getenv("GRAPH_API_TOKEN")
headers = {
    "Authorization": f"Bearer {access_token}"
}
response = requests.get("https://graph.microsoft.com/v1.0/me/messages", headers=headers)
emails = response.json()
for email in emails.get("value", []):
    print(email["subject"])

## Using Third-Party E-mail Services (Bulk & Transactional E-mail)

When automation moves beyond “send a few notifications to myself,” **SMTP via Gmail or Outlook quickly stops being enough**.  
This is where **third-party transactional and bulk e-mail services** come in.

These services sit between your application and the global e-mail infrastructure and are designed for **reliability, scale, and deliverability**.

---

### Why Third-Party E-mail Services Exist

#### Problems with direct SMTP (Gmail / Outlook)

- Strict daily sending limits
- Aggressive spam filtering
- Accounts get temporarily blocked without warning
- No delivery analytics
- No retry or bounce handling
- Poor support for bulk or automated mail

SMTP is fine for:
- alerts
- reports
- personal automation

It is **not** designed for:
- newsletters
- student notifications at scale
- system-generated mail to many recipients

---

### What Third-Party Providers Offer

Typical capabilities:

- High deliverability (IP reputation management)
- Bulk sending without account bans
- Bounce and spam complaint handling
- Rate limiting and retries
- Web dashboards and logs
- Dedicated APIs (HTTP-based, not SMTP)
- Webhooks for delivery events

They turn “send an email” into a **reliable infrastructure service**.

---

### Common Providers (as of 2025-2026)

Most are paid services with free tiers for low-volume use. As a student or small project, free tiers are often sufficient.

#### Transactional / Developer-Focused

- **SendGrid** (Twilio)  
  https://sendgrid.com  

- **Mailgun**  
  https://www.mailgun.com  

- **Amazon SES**  
  https://aws.amazon.com/ses  

- **Postmark**  
  https://postmarkapp.com  

---

#### Newsletter / Marketing-Oriented

- **Mailchimp**  
  https://mailchimp.com  

- **Brevo (formerly Sendinblue)**  
  https://www.brevo.com  

These are usually **not ideal for pure automation scripts**, but good for managed campaigns.

---

### Typical Architecture with a Third-Party Provider

Instead of SMTP:

```
Python Script
    ↓ HTTPS (API call)
Email Provider API
    ↓
Recipient Mail Server
```

Key difference:
- You authenticate **your application**, not a mailbox
- Usually via API key or OAuth
- No user login involved

---

### Advantages

- Much higher sending limits  
- Better spam avoidance  
- Clear error reporting  
- Delivery and open tracking  
- Designed for automation  
- Safer separation of concerns  

---

### Disadvantages

- External dependency  
- Account setup required  
- Possible costs at scale  
- API keys must be protected  
- Overkill for small scripts  

For teaching:
- Conceptually important
- Often demonstrated, not required

---

### Security Model (Conceptual)

- You create an **API key** in provider dashboard
- API key:
  - Identifies your application
  - Grants limited permissions
- Stored as environment variable (same pattern as SMTP)
- Can be rotated or revoked

This is simpler than OAuth, but still requires care.

---

### Pseudocode Example (Provider-Agnostic)

```text
LOAD API_KEY from environment

BUILD email payload:
    from
    to
    subject
    body (text / HTML)
    attachments (optional)

SEND HTTPS POST request to provider API
    include API_KEY in headers
    include payload as JSON

IF response == success:
    log message ID
ELSE:
    log error
    retry or abort
```

---

### Conceptual Python-Style Pseudocode

```python
api_key = getenv("EMAIL_API_KEY")

email = {
    "from": "noreply@example.com",
    "to": ["user1@example.com", "user2@example.com"],
    "subject": "Automated Notification",
    "text": "Your task has completed successfully."
}

response = POST(
    url="https://api.emailprovider.com/send",
    headers={"Authorization": f"Bearer {api_key}"},
    json=email
)

if response.ok:
    log("Email sent")
else:
    log("Error:", response.error)
```

---

### When You Should Consider a Third-Party Service

Use one if **any** of these are true:

- You send e-mails to many recipients
- You send e-mails regularly or automatically
- You need delivery reliability
- You want logs and diagnostics
- You do not want to risk personal accounts being blocked

---

### When You Should Not Use One

- One-off scripts
- Personal notifications
- Teaching basic automation
- Offline or restricted environments

SMTP remains the best first step.

---

### Key Conceptual Takeaway

- SMTP teaches *how e-mail works*
- Gmail / Outlook APIs teach *OAuth and security*
- Third-party services teach *production-scale thinking*

## Summary and Challenges Ahead

- File system automation is simple and robust
- SMTP e-mail sending is reliable and widely supported
- Full mailbox APIs introduce significant security complexity


## Independent Project Ideas

- File archiver with e-mail notifications
- Automated report sender
- Attachment downloader (Gmail API)
